# Visualize Unlabeled Acoustic Embeddings (UMAP+t-SNE 20D + HDBSCAN)

This notebook creates static and interactive 3D visualizations using the first 3 components from 20D UMAP/t-SNE embeddings, colored by HDBSCAN labels.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from os import environ

import umap
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import hdbscan
import plotly.express as px

In [ ]:
env_dir = environ.get("POSIDONIA_DATASET_DIR")
DATASET_DIR = Path(env_dir) if env_dir else Path("D:/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/dataset")

EMBEDDINGS_PATH = DATASET_DIR / "npy_files" / "unlabeled_embeddings.npy"
MANIFEST_PATH = DATASET_DIR / "unlabeled_manifest.csv"

if not EMBEDDINGS_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing files:\n"
        f"  embeddings: {EMBEDDINGS_PATH}\n"
        f"  manifest:   {MANIFEST_PATH}\n"
        f"Set POSIDONIA_DATASET_DIR to the correct dataset folder."
    )

embeddings = np.load(str(EMBEDDINGS_PATH))
manifest_df = pd.read_csv(str(MANIFEST_PATH))

print(f"Loaded {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
print(f"Manifest rows: {len(manifest_df)}")

In [ ]:
output_dir = EMBEDDINGS_PATH.parent
umap_tsne_output_dir = output_dir / "umap_and_tsne_20d"
umap_tsne_output_dir.mkdir(parents=True, exist_ok=True)

umap_file = umap_tsne_output_dir / "umap_embeddings_20d.npy"
tsne_file = umap_tsne_output_dir / "tsne_subset_embeddings_20d.npy"
tsne_idx_file = umap_tsne_output_dir / "tsne_subset_indices_20d.npy"

if umap_file.exists() and tsne_file.exists() and tsne_idx_file.exists():
    print("Loading pre-computed UMAP(20D) and t-SNE(20D)...")
    umap_results = np.load(str(umap_file))
    tsne_results = np.load(str(tsne_file))
    tsne_idx = np.load(str(tsne_idx_file))
else:
    print("Computing PCA(256) -> UMAP(20D) and t-SNE(20D subset)...")
    embeddings_fp32 = embeddings.astype(np.float32, copy=False)
    embeddings_pca_256 = PCA(n_components=256, random_state=42).fit_transform(embeddings_fp32)

    umap_results = umap.UMAP(
        n_components=20,
        random_state=42,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        low_memory=True
    ).fit_transform(embeddings_pca_256)

    TSNE_VIS_LIMIT = 50000
    rng = np.random.default_rng(42)
    tsne_idx = rng.choice(embeddings.shape[0], size=min(TSNE_VIS_LIMIT, embeddings.shape[0]), replace=False)
    tsne_results = TSNE(n_components=20, random_state=42, perplexity=30).fit_transform(embeddings_pca_256[tsne_idx])

    np.save(str(umap_file), umap_results)
    np.save(str(tsne_file), tsne_results)
    np.save(str(tsne_idx_file), tsne_idx)

print("UMAP(20D) shape:", umap_results.shape)
print("t-SNE(20D) shape:", tsne_results.shape)

In [ ]:
HDBSCAN_MIN_CLUSTER_SIZE = 80
HDBSCAN_MIN_SAMPLES = 10

umap_labels_file = umap_tsne_output_dir / f"umap_20d_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"
tsne_labels_file = umap_tsne_output_dir / f"tsne_20d_subset_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"

if umap_labels_file.exists():
    umap_labels = np.load(str(umap_labels_file))
else:
    umap_labels = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    ).fit_predict(umap_results)
    np.save(str(umap_labels_file), umap_labels)

if tsne_labels_file.exists():
    tsne_labels = np.load(str(tsne_labels_file))
else:
    tsne_labels = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    ).fit_predict(tsne_results)
    np.save(str(tsne_labels_file), tsne_labels)

print("UMAP cluster labels shape:", umap_labels.shape)
print("t-SNE cluster labels shape:", tsne_labels.shape)
print("UMAP clusters:", np.sum(np.unique(umap_labels) >= 0), "| noise:", np.sum(umap_labels == -1))
print("t-SNE clusters:", np.sum(np.unique(tsne_labels) >= 0), "| noise:", np.sum(tsne_labels == -1))

## Visualize

In [ ]:
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    umap_results[:, 0], umap_results[:, 1], umap_results[:, 2],
    c=umap_labels, cmap="gist_rainbow", s=5, alpha=0.35
)
ax.set_title("UMAP(20D -> first 3 dims) HDBSCAN Clustering")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_zlabel("UMAP 3")
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    tsne_results[:, 0], tsne_results[:, 1], tsne_results[:, 2],
    c=tsne_labels, cmap="gist_rainbow", s=5, alpha=0.35
)
ax.set_title("t-SNE(20D subset -> first 3 dims) HDBSCAN Clustering")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_zlabel("t-SNE 3")
plt.tight_layout()
plt.show()

In [ ]:
plot_df_umap = pd.DataFrame({
    'x': umap_results[:, 0],
    'y': umap_results[:, 1],
    'z': umap_results[:, 2],
    'cluster': umap_labels.astype(str),
    'audio_file': manifest_df['audio_path'].apply(lambda x: Path(x).name) if 'audio_path' in manifest_df.columns else 'n/a'
})

fig_umap = px.scatter_3d(
    plot_df_umap,
    x='x', y='y', z='z',
    color='cluster',
    hover_data=['cluster', 'audio_file'],
    title='UMAP 20D (first 3 dims) with HDBSCAN Clusters',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'},
    opacity=0.75
)
fig_umap.update_traces(marker=dict(size=3))
fig_umap.update_layout(height=800)
fig_umap.show()

In [ ]:
audio_series = (
    manifest_df.iloc[tsne_idx]['audio_path'].apply(lambda x: Path(x).name).values
    if 'audio_path' in manifest_df.columns else np.array(['n/a'] * len(tsne_results))
)

plot_df_tsne = pd.DataFrame({
    'x': tsne_results[:, 0],
    'y': tsne_results[:, 1],
    'z': tsne_results[:, 2],
    'cluster': tsne_labels.astype(str),
    'audio_file': audio_series
})

fig_tsne = px.scatter_3d(
    plot_df_tsne,
    x='x', y='y', z='z',
    color='cluster',
    hover_data=['cluster', 'audio_file'],
    title='t-SNE 20D subset (first 3 dims) with HDBSCAN Clusters',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2', 'z': 't-SNE 3'},
    opacity=0.75
)
fig_tsne.update_traces(marker=dict(size=3))
fig_tsne.update_layout(height=800)
fig_tsne.show()